# Causal language model

Causal language modeling (AKA autoregressive modeling) predicts the token in a sequence of tokens, and the model can only attend to tokens on the left. This means the model cannot see future tokens. Used by decode models like GPT. this approach predicts the next token based on all previous tokens in the left to predict the next token.

In [5]:
!pip install transformers datasets evaluate

## Load ELI5 dataset

Start by loading the first 5000 examples from ELI5-Category dataset with the hugging face datasets libary.

In [6]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Split the dataset's <b>train</b> split into a train and test set with the <b>train_text_split</b> method:

In [7]:
eli5 = eli5.train_test_split(test_size=0.2)
eli5 = eli5.flatten()
eli5["train"][0]

{'q_id': '7cobrx',
 'title': "Why does the Salt in McDonald's Salt Packets Taste Different from the Salt They're Prepared with?",
 'selftext': '',
 'category': 'Chemistry',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dpreaaz'],
 'answers.text': ["It's all just salt. The difference is the size of the grains. The salt that is used when making french fries has super-fine crystals - it's almost a powder while the salt in the packets is more like a traditional table salt. The size of the salt crystals greatly influences how it impacts your taste buds."],
 'answers.score': [10],
 'answers.text_urls': [[]],
 'title_urls': ['url'],
 'selftext_urls': ['url']}

While this looks like a lot, we are only interested in the <b>text</b> field. What's cool about language modeling tasks is you don't need labels(also knows as an unsupervised task) because the next word is the label.

## Preprocess
The next step is to load a DIstilGPT2 tokenizer to process the <b>text</b> subfield.

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

You’ll notice from the example above, the text field is actually nested inside answers. This means you’ll need to extract the text subfield from its nested structure with the flatten method.

Each subfield is now a separate column as indicated by the answers prefix, and the text field is a list now. Instead of tokenizing each sentence separately, convert the list to a string so we can jointly tokenize them.

Here is a first preprocessing function to join the list of strings for each example and tokenize the result:

In [9]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

To apply this preprocessing function over the entire dataset, lets use the 🤗 Datasets map method. You can speed up the map function by setting batched=True to process multiple elements of the dataset at once, and increasing the number of processes with num_proc. Remove any columns you don’t need:

In [10]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1384 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1320 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1133 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2166 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1881 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3376 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1140 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2135 > 1024). Running this sequence through the model will result in indexing errors


In [11]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

Apply the <b>group_text</b> function over the entire dataset:

In [12]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Now create a batch of examplesusing <b>DataCollatorForLanguageModeling</b>. Its more efficient to dynamically pad the sentence to the longest length in batch during collation, instead of padding the whole dataset to the maximum length.

Use the end-of-sequence token as the padding token and set <b>mlm=False</b>. This will use the inputs as labels shifted to the right by one element:

In [13]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

Now we are ready to start training our model! Load DistilGPT2 with AutoModelForCausalLM:

In [15]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

At this point, only three steps remain:

- Define your training hyperparameters in <b>TrainingArguments</b>. The only required parameter is output_dir which specifies where to save your model. You’ll push this model to the Hub by setting push_to_hub=True (you need to be signed in to Hugging Face to upload your model).
- Pass the training arguments to <b>Trainer</b> along with the model, datasets, and data collator.
- Call <b>train()</b> to finetune your model.

In [ ]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()